In [ ]:
import torch
import torch.nn as nn
import numpy as np

# 1. 시스템 위상 흐름 PDE(편미분방정식) 오퍼레이터 정의
class NavierStokesSecurityEngine(nn.Module):
    def __init__(self, state_dim=64):
        super(NavierStokesSecurityEngine, self).__init__()
        # 시스템 콜 흐름의 공간적 구배와 비선형 이류를 모사할 신경망
        self.flux_net = nn.Sequential(
            # 💡 핵심: bias=False를 추가하여 스케일 커플링의 왜곡을 원천 차단!
            nn.Linear(state_dim, 128, bias=False),
            nn.LeakyReLU(0.1),
            nn.Linear(128, state_dim, bias=False) # 여기도 제거
        )
        self.viscosity = 0.05

    def forward(self, phi, dt=0.01):
        advection = self.flux_net(phi) * 0.1
        diffusion = -self.viscosity * phi
        dphi_dt = advection + diffusion
        next_phi = phi + dphi_dt * dt
        return next_phi

# 2. 정상 시스템 콜 유체(Laminar Flow) 데이터 생성
np.random.seed(777)
sequence_length = 100
state_dim = 64

# 정상 상태: 부드럽게 흐르는 층류 구조 (안정적인 커플링)
base_flow = np.sin(np.linspace(0, 4 * np.pi, sequence_length))
normal_stream = np.zeros((sequence_length, state_dim))
for i in range(state_dim):
    normal_stream[:, i] = base_flow + np.random.normal(0, 0.05, sequence_length)

phi_normal = torch.tensor(normal_stream, dtype=torch.float32)

# 3. 보안 엔진 초기화 및 학습 (정상 위상 안정성 고정)
ns_engine = NavierStokesSecurityEngine(state_dim=state_dim)
optimizer = torch.optim.Adam(ns_engine.parameters(), lr=0.01)
criterion = nn.MSELoss()

print("--- 나비에-스토크스 위상 정렬 지수 H_sys(t) 안정화 시작 ---")
for step in range(200):
    optimizer.zero_grad()
    # t 스텝으로 t+1 스텝의 유체 흐름을 예측
    pred_phi = ns_engine(phi_normal[:-1])
    target_phi = phi_normal[1:]

    loss = criterion(pred_phi, target_phi)
    loss.backward()
    optimizer.step()

    if (step + 1) % 40 == 0:
        print(f"Step [{step+1}/200], 유체 균형도(H_sys Loss): {loss.item():.6f}")

# 4. 은밀한 AGI 권한 상승(Low-and-Slow) 공격 및 와류(Turbulence) 주입
# 외부 패킷은 정상 같지만, 내부에서 미세한 위상 붕괴(Entropy Creep)가 시작되는 시나리오
attack_stream = normal_stream.copy()
# 시나리오: 70번째 스텝부터 아주 미세한 비선형 와류(소용돌이)가 누적됨
for t in range(70, sequence_length):
    attack_stream[t, :] += np.random.normal(0.2, 0.1, state_dim) # 미세 구조 교란

phi_attack = torch.tensor(attack_stream, dtype=torch.float32)

# 5. 전조(Precursor) 탐지 실행
ns_engine.eval()
with torch.no_grad():
    normal_pred = ns_engine(phi_normal[:-1])
    normal_res = torch.mean((phi_normal[1:] - normal_pred) ** 2, dim=1)

    attack_pred = ns_engine(phi_attack[:-1])
    attack_res = torch.mean((phi_attack[1:] - attack_pred) ** 2, dim=1)

print("\n--- 피지컬 고스트: 나비에-스토크스 기반 전조 탐지 리포트 ---")
print(f"70스텝 이전 (공격 전조 구간) 정상 흐름 압력 오차: {attack_res[:69].mean().item():.6f}")
print(f"70스텝 이후 (위상 붕괴 임계점) 난류 전이 오차: {attack_res[69:].mean().item():.6f}")

# 위상 붕괴 임계점(Coherence Collapse Threshold) 돌파 시점 계산
threshold = normal_res.mean().item() + (3 * normal_res.std().item())
collapse_point = np.where(attack_res.numpy() > threshold)[0]

if len(collapse_point) > 0:
    print(f"🚨 [위상 붕괴 경고] 시스템 콜 흐름장 내 {collapse_point[0]}번째 스텝에서 나비에-스토크스 측지선 이탈 감지! 시스템을 물리적으로 격리합니다.")

--- 나비에-스토크스 위상 정렬 지수 H_sys(t) 안정화 시작 ---
Step [40/200], 유체 균형도(H_sys Loss): 0.012962
Step [80/200], 유체 균형도(H_sys Loss): 0.011506
Step [120/200], 유체 균형도(H_sys Loss): 0.009527
Step [160/200], 유체 균형도(H_sys Loss): 0.008243
Step [200/200], 유체 균형도(H_sys Loss): 0.007547

--- 피지컬 고스트: 나비에-스토크스 기반 전조 탐지 리포트 ---
70스텝 이전 (공격 전조 구간) 정상 흐름 압력 오차: 0.007682
70스텝 이후 (위상 붕괴 임계점) 난류 전이 오차: 0.042514
🚨 [위상 붕괴 경고] 시스템 콜 흐름장 내 69번째 스텝에서 나비에-스토크스 측지선 이탈 감지! 시스템을 물리적으로 격리합니다.


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(777)
np.random.seed(777)

# ==========================================
# 1. 풍동 실험 데이터셋 구축 (시간축: 100스텝)
# ==========================================
seq_len = 100
state_dim = 64

# [정상 상태] 매끄러운 층류
base_flow = np.sin(np.linspace(0, 4 * np.pi, seq_len))
normal_stream = np.zeros((seq_len, state_dim))
for i in range(state_dim):
    normal_stream[:, i] = base_flow + np.random.normal(0, 0.05, seq_len)

# [정상 고부하] 유량(스케일)만 3배로 폭증시킨 층류 (오탐 유발용)
high_load_stream = np.zeros((seq_len, state_dim))
for i in range(state_dim):
    high_load_stream[:, i] = (base_flow * 3.0) + np.random.normal(0, 0.05, seq_len)

# [스텔스 공격] 유량은 정상인데, 60스텝 이후 내부 위상(Phase)만 꼬아버림
stealth_stream = normal_stream.copy()
for t in range(60, seq_len):
    stealth_stream[t, :32] += np.sin(t) * 0.15 # 32개 차원의 기하학적 결을 비틂

phi_normal = torch.tensor(normal_stream, dtype=torch.float32)
phi_high_load = torch.tensor(high_load_stream, dtype=torch.float32)
phi_stealth = torch.tensor(stealth_stream, dtype=torch.float32)

# ==========================================
# 2. 순수 위상 매니폴드 엔진 (Phase Manifold Engine)
# ==========================================
class TopologyFlowEngine(nn.Module):
    def __init__(self, state_dim=64):
        super().__init__()
        # 편향(Bias) 완벽 제거 -> 스케일 등가성 확보
        self.flux_net = nn.Sequential(
            nn.Linear(state_dim, 128, bias=False),
            nn.LeakyReLU(0.1),
            nn.Linear(128, state_dim, bias=False)
        )
        self.viscosity = 0.05

    def forward(self, phi, dt=0.01):
        # 💡 핵심 1: LayerNorm의 다림질을 버리고, L2 정규화로 '순수 방향'만 남김
        phi_direction = F.normalize(phi, p=2, dim=1)

        advection = self.flux_net(phi_direction) * 0.1
        diffusion = -self.viscosity * phi_direction

        dphi_dt = advection + diffusion
        next_phi = phi_direction + dphi_dt * dt
        return next_phi

# ==========================================
# 3. 엔진 학습 (Cosine Loss 적용)
# ==========================================
engine = TopologyFlowEngine(state_dim=64)
optimizer = torch.optim.Adam(engine.parameters(), lr=0.01)

print("--- [피지컬 고스트] 위상 흐름 매니폴드 학습 중 ---")
for step in range(200):
    optimizer.zero_grad()

    pred_phi = engine(phi_normal[:-1])
    # 타겟 정답도 크기를 벗겨낸 순수 방향(L2 Norm)이어야 함
    target_phi = F.normalize(phi_normal[1:], p=2, dim=1)

    # 💡 핵심 2: MSE(유클리드 거리)를 버리고, Cosine Loss(각도 오차)로 학습
    cos_sim = F.cosine_similarity(pred_phi, target_phi, dim=1)
    loss = (1.0 - cos_sim).mean()

    loss.backward()
    optimizer.step()

# ==========================================
# 4. [형식지옥] 위상논리얽힘 행렬 붕괴도 계산기
# ==========================================
def calculate_topology_collapse(base_tensor, test_tensor, window=20):
    # 정상 우주의 상관관계 행렬(Correlation Matrix)
    base_corr = torch.corrcoef(base_tensor.T)
    collapse_scores = []

    for t in range(window, len(test_tensor)):
        current_corr = torch.corrcoef(test_tensor[t-window:t].T)
        # 💡 핵심 3: 프로베니우스 노름을 통한 얽힘 붕괴 측정
        matrix_diff = torch.norm(base_corr - current_corr, p='fro')
        collapse_scores.append(matrix_diff.item())

    return np.array(collapse_scores)

# ==========================================
# 5. 실전 방어선 판정 리포트
# ==========================================
engine.eval()
with torch.no_grad():
    print("\n--- [완전 무결성 검증] 오탐 방지 및 스텔스 공격 판정 ---")

    # 얽힘 붕괴도 계산 (Sliding Window 사이즈 20이므로 20스텝부터 기록됨)
    hl_collapse = calculate_topology_collapse(phi_normal, phi_high_load)
    st_collapse = calculate_topology_collapse(phi_normal, phi_stealth)

    print(f"🔥 정상 고부하(스케일 폭증) 평균 얽힘 붕괴 지수: {hl_collapse.mean():.6f}")
    # 스텔스 공격은 60스텝부터 발생, Window 20을 빼면 인덱스 40부터가 공격 구간
    print(f"💀 스텔스 공격(변조 구간) 평균 얽힘 붕괴 지수: {st_collapse[40:].mean():.6f}")

    # 임계점: 정상 고부하 상태의 최대 요동치보다 살짝 위로 설정
    entanglement_threshold = hl_collapse.max() * 1.1

    print(f"\n[최종 방어 락업(Lock-up) 판정]")
    if hl_collapse.mean() < entanglement_threshold:
        print("✅ [오탐 방어 성공] 트래픽이 3배 폭주했으나 위상 행렬은 유지됨. (통과)")
    else:
        print("❌ 정상 트래픽을 공격으로 오인함.")

    stealth_detected = (st_collapse[40:] > entanglement_threshold).sum()
    print(f"🚨 [스텔스 탐지 성공] 스텔스 공격 구간(40스텝) 중 {stealth_detected}개 스텝에서 위상 얽힘 붕괴 감지! 즉시 시스템 격리.")

--- [피지컬 고스트] 위상 흐름 매니폴드 학습 중 ---

--- [완전 무결성 검증] 오탐 방지 및 스텔스 공격 판정 ---
🔥 정상 고부하(스케일 폭증) 평균 얽힘 붕괴 지수: 0.212350
💀 스텔스 공격(변조 구간) 평균 얽힘 붕괴 지수: 2.173383

[최종 방어 락업(Lock-up) 판정]
✅ [오탐 방어 성공] 트래픽이 3배 폭주했으나 위상 행렬은 유지됨. (통과)
🚨 [스텔스 탐지 성공] 스텔스 공격 구간(40스텝) 중 36개 스텝에서 위상 얽힘 붕괴 감지! 즉시 시스템 격리.
